In [ ]:
!pip install --quiet category_encoders
!pip install --quiet scikit-plot

     |████████████████████████████████| 81kB 2.2MB/s 


In [ ]:
%cd "/content/drive/My Drive/ICFernando/SEER Remodelado"

/content/drive/My Drive/ICFernando/SEER Remodelado


In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from category_encoders.cat_boost import CatBoostEncoder
from imblearn.over_sampling import RandomOverSampler

/usr/local/lib/python3.6/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm
/usr/local/lib/python3.6/dist-packages/sklearn/externals/six.py:31: FutureWarning: The module is deprecated in version 0.21 and will be removed in version 0.23 since we've dropped support for Python 2.7. Please rely on the official version of six (https://pypi.org/project/six/).
  "(https://pypi.org/project/six/).", FutureWarning)
/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:144: FutureWarning: The sklearn.neighbors.base module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.neighbors. Anything that cannot be imported from sklearn.neighbors is now part of the private API.
  warnings.warn(message, FutureWarning)


In [ ]:
def arrumar_linf_exa(df):
    df.loc[df['LINF_EXA_POS'] > df['LINF_EXA'], 'LINF_EXA_POS'] = df.loc[df['LINF_EXA_POS'] > df['LINF_EXA'], 'LINF_EXA']
    return df

In [ ]:
targets = ['CURADO', 'SOBRE_COM_CANCER', 'TEMPO_SOBRE']
tipos_tumor = ['OTHER', 'BREAST', 'MALEGEN', 'RESPIR', 'COLRECT', 'LYMYLEUK', 'DIGOTHR', 'URINARY', 'FEMGEN', 'ALL']

In [ ]:
for target in targets:
    df_total = pd.read_csv('./bases/{0}/{0}.csv'.format(target.lower()))
    print('Base principal carregada')
    for tipo in tipos_tumor:
        print(tipo, target)

        if tipo != 'ALL':
            df = df_total[df_total['TIPO_TUMOR'] == tipo].drop(columns='TIPO_TUMOR')
        else:
            df = df_total.drop(columns='TIPO_TUMOR')
            del df_total
        print('Tipo de tumor selecionado')
        
        if target == 'TEMPO_SOBRE':
            df[target] = df[target].fillna(0)
        continuas = ['HISTORICO_TUMOR', 'IDADE_DIAG', 'HISTORICO_MALIGNO', 'HISTORICO_BENIGNO', 'LINF_EXA', 'LINF_EXA_POS']
        categoricas = df.columns.drop(labels=continuas + [target])

        if target == 'TEMPO_SOBRE':
            X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=target), df[target], test_size=(1/4), random_state=1, shuffle=True)
            del df
            X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=(1/3), random_state=1, shuffle=True)
        else:
            X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=target), df[target], test_size=(1/4), random_state=1, shuffle=True, stratify=df[target])
            del df
            X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=(1/3), random_state=1, shuffle=True, stratify=y_train)
        print('Dividido em treino, validação e teste')

        print(y_train.value_counts())
        print(y_valid.value_counts())
        print(y_test.value_counts())

        imp_continuas = SimpleImputer(strategy='mean')
        X_train[continuas] = imp_continuas.fit_transform(X_train[continuas])
        X_valid[continuas] = imp_continuas.transform(X_valid[continuas])
        X_test[continuas] = imp_continuas.transform(X_test[continuas])
        print('Preencher nulo de contínuas feito')

        valores_freq = []
        categoricas_erro = []
        for coluna in categoricas:
            try:
                valor_freq = X_train[coluna].mode()[0]
            except IndexError:
                print('ERRO AO TRATAR COLUNA: {0}'.format(coluna.upper()))
                print(X_train[coluna].mode())
                print(X_valid[coluna].mode())
                print(X_test[coluna].mode())
                X_train.drop(columns=coluna, inplace=True)
                X_valid.drop(columns=coluna, inplace=True)
                X_test.drop(columns=coluna, inplace=True)
                categoricas_erro.append(coluna)
                continue
            valores_freq.append([coluna, valor_freq])
            X_train[coluna].fillna(valor_freq, inplace=True)
            X_valid[coluna].fillna(valor_freq, inplace=True)
            X_test[coluna].fillna(valor_freq, inplace=True)
        categoricas = categoricas.drop(labels=categoricas_erro)
        print('Preencher nulo de categóricas feito')

        X_train = arrumar_linf_exa(X_train)
        X_valid = arrumar_linf_exa(X_valid)
        X_test = arrumar_linf_exa(X_test)
        print('Arrumar linf_exa feito')

        encoder_categorica = CatBoostEncoder(verbose=0, cols=categoricas, drop_invariant=False,
                                            return_df=True, handle_unknown='value',
                                            handle_missing='value', random_state=1, sigma=0.1, a=20)
        X_train = encoder_categorica.fit_transform(X_train, y_train)
        X_valid = encoder_categorica.transform(X_valid)
        X_test = encoder_categorica.transform(X_test)
        print('Tratamento de categóricas feito')

        transformadores = [('continuas', StandardScaler(), continuas)]
        ct = ColumnTransformer(transformadores, sparse_threshold=0, n_jobs=1, verbose=True, remainder='passthrough')
        X_train = ct.fit_transform(X_train)
        X_valid = ct.transform(X_valid)
        X_test = ct.transform(X_test)
        print('Tratamento de contínuas feito')
        
        if target != 'TEMPO_SOBRE':
            ros = RandomOverSampler(random_state=1)
            X_train, y_train = ros.fit_resample(X_train, y_train)
            print('Oversampling feito')


        with open('./bases/{0}/{1}/encoders.pkl'.format(target.lower(), tipo), 'wb') as arq:
            pickle.dump([imp_continuas, valores_freq, categoricas_erro, encoder_categorica, ct], arq)
        print('Salvo {0}'.format('Encoders'))

        with open('./bases/{0}/{1}/X_train.pkl'.format(target.lower(), tipo), 'wb') as arq:
            pickle.dump(X_train, arq)
        print('Salvo {0}'.format('X_train'))
        
        with open('./bases/{0}/{1}/X_valid.pkl'.format(target.lower(), tipo), 'wb') as arq:
            pickle.dump(X_valid, arq)
        print('Salvo {0}'.format('X_valid'))
        
        with open('./bases/{0}/{1}/X_test.pkl'.format(target.lower(), tipo), 'wb') as arq:
            pickle.dump(X_test, arq)
        print('Salvo {0}'.format('X_test'))

        with open('./bases/{0}/{1}/y_train.pkl'.format(target.lower(), tipo), 'wb') as arq:
            pickle.dump(y_train, arq)
        print('Salvo {0}'.format('y_train'))

        with open('./bases/{0}/{1}/y_valid.pkl'.format(target.lower(), tipo), 'wb') as arq:
            pickle.dump(y_valid, arq)
        print('Salvo {0}'.format('y_valid'))

        with open('./bases/{0}/{1}/y_test.pkl'.format(target.lower(), tipo), 'wb') as arq:
            pickle.dump(y_test, arq)
        print('Salvo {0}'.format('y_test'))

        del X_train, X_valid, X_test, y_train, y_valid, y_test

Base principal carregada
OTHER CURADO
Tipo de tumor selecionado
Dividido em treino, validação e teste
0    568964
1    405700
Name: CURADO, dtype: int64
0    284483
1    202850
Name: CURADO, dtype: int64
0    284483
1    202850
Name: CURADO, dtype: int64
Preencher nulo de contínuas feito
Preencher nulo de categóricas feito
Arrumar linf_exa feito
Tratamento de categóricas feito
[ColumnTransformer] ..... (1 of 2) Processing continuas, total=   0.1s
[ColumnTransformer] ..... (2 of 2) Processing remainder, total=   0.0s
Tratamento de contínuas feito


/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


Oversampling feito
Salvo Encoders
Salvo X_train
Salvo X_valid
Salvo X_test
Salvo y_train
Salvo y_valid
Salvo y_test
BREAST CURADO
Tipo de tumor selecionado
Dividido em treino, validação e teste
1    504637
0    343951
Name: CURADO, dtype: int64
1    252319
0    171976
Name: CURADO, dtype: int64
1    252319
0    171976
Name: CURADO, dtype: int64
Preencher nulo de contínuas feito
ERRO AO TRATAR COLUNA: CIRURGIA_LINF
Series([], dtype: float64)
Series([], dtype: float64)
Series([], dtype: float64)
Preencher nulo de categóricas feito
Arrumar linf_exa feito
Tratamento de categóricas feito
[ColumnTransformer] ..... (1 of 2) Processing continuas, total=   0.1s
[ColumnTransformer] ..... (2 of 2) Processing remainder, total=   0.0s
Tratamento de contínuas feito


/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


Oversampling feito
Salvo Encoders
Salvo X_train
Salvo X_valid
Salvo X_test
Salvo y_train
Salvo y_valid
Salvo y_test
MALEGEN CURADO
Tipo de tumor selecionado
Dividido em treino, validação e teste
1    421642
0    269680
Name: CURADO, dtype: int64
1    210821
0    134841
Name: CURADO, dtype: int64
1    210822
0    134840
Name: CURADO, dtype: int64
Preencher nulo de contínuas feito
Preencher nulo de categóricas feito
Arrumar linf_exa feito
Tratamento de categóricas feito
[ColumnTransformer] ..... (1 of 2) Processing continuas, total=   0.1s
[ColumnTransformer] ..... (2 of 2) Processing remainder, total=   0.0s
Tratamento de contínuas feito


...

Oversampling feito
Salvo Encoders
Salvo X_train
Salvo X_valid
Salvo X_test
Salvo y_train
Salvo y_valid
Salvo y_test
FEMGEN SOBRE_COM_CANCER
Tipo de tumor selecionado
Dividido em treino, validação e teste
0.0    194793
1.0     79145
Name: SOBRE_COM_CANCER, dtype: int64
0.0    97397
1.0    39572
Name: SOBRE_COM_CANCER, dtype: int64
0.0    97397
1.0    39573
Name: SOBRE_COM_CANCER, dtype: int64
Preencher nulo de contínuas feito
Preencher nulo de categóricas feito
Arrumar linf_exa feito
Tratamento de categóricas feito
[ColumnTransformer] ..... (1 of 2) Processing continuas, total=   0.0s
[ColumnTransformer] ..... (2 of 2) Processing remainder, total=   0.0s
Tratamento de contínuas feito


/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


Oversampling feito
Salvo Encoders
Salvo X_train
Salvo X_valid
Salvo X_test
Salvo y_train
Salvo y_valid
Salvo y_test
ALL SOBRE_COM_CANCER
Tipo de tumor selecionado
Dividido em treino, validação e teste
0.0    2770905
1.0    1483481
Name: SOBRE_COM_CANCER, dtype: int64
0.0    1385453
1.0     741740
Name: SOBRE_COM_CANCER, dtype: int64
0.0    1385452
1.0     741741
Name: SOBRE_COM_CANCER, dtype: int64
Preencher nulo de contínuas feito
Preencher nulo de categóricas feito
Arrumar linf_exa feito
Tratamento de categóricas feito
[ColumnTransformer] ..... (1 of 2) Processing continuas, total=   0.6s
[ColumnTransformer] ..... (2 of 2) Processing remainder, total=   0.0s
Tratamento de contínuas feito


/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


Oversampling feito
Salvo Encoders
Salvo X_train
Salvo X_valid
Salvo X_test
Salvo y_train
Salvo y_valid
Salvo y_test
Base principal carregada
OTHER TEMPO_SOBRE
Tipo de tumor selecionado
Dividido em treino, validação e teste
0.0      70164
1.0      32042
2.0      24392
3.0      20100
4.0      18236
         ...  
490.0       31
503.0       30
492.0       29
494.0       27
484.0       26
Name: TEMPO_SOBRE, Length: 504, dtype: int64
0.0      34650
1.0      16266
2.0      12126
3.0      10152
4.0       9284
         ...  
484.0       16
500.0       15
492.0       11
503.0       11
491.0       10
Name: TEMPO_SOBRE, Length: 504, dtype: int64
0.0      35128
1.0      16032
2.0      12217
3.0       9980
4.0       8984
         ...  
500.0       15
502.0       15
492.0       14
503.0       13
480.0       11
Name: TEMPO_SOBRE, Length: 504, dtype: int64
Preencher nulo de contínuas feito
Preencher nulo de categóricas feito
Arrumar linf_exa feito
Tratamento de categóricas feito
[ColumnTransformer] ..